# v2 → v6 일관성 · 이동 패턴 분석

같은 100명의 페르소나가 컨텍스트가 v2 → v3 → v4 → v5 → v6 으로 변하는 동안 어떻게 이동했는지, 그리고 LLM이 각 단계의 지침에 얼마나 충실히 반응했는지 분석한다.

## 파이프라인
1. CSV 에서 (모델 × v2-v6 × 동일 페르소나) 추출 → 페르소나 카드와 합쳐 청크 10개로 분할
2. Claude Code Agent 10개 병렬 dispatch — 각 청크 평가 (1~5 fidelity, trajectory 분류, narrative)
3. 결과 집계 → `doc/v2_v6_consistency_analysis_{model}.md` + `.json`
4. 슬라이드 placeholder에 결과 반영

In [ ]:
from pathlib import Path
import json
import pandas as pd

BASE = Path.cwd()
RESULTS = BASE / "vote_results_all.csv"
INTERESTS = BASE / "interests" / "personas_gpt-5.4-mini_all_interests.tsv"
EVAL_DIR = BASE / "eval"
EVAL_DIR.mkdir(exist_ok=True)

GPT_MODEL = "openai/gpt-5.4-mini"
HAIKU_MODEL = "anthropic/claude-haiku-4-5-20251001"

df = pd.read_csv(RESULTS, encoding="utf-8-sig")
interests = pd.read_csv(INTERESTS, sep="\t", encoding="utf-8-sig")
print("전체 행:", len(df))
print("interests 행:", len(interests))

In [ ]:
# 페르소나당 (model, version) → (vote, reason) 매트릭스 만들기
def build_chunks(model: str, versions: list[str], chunks_n: int = 10) -> list[list[dict]]:
    """각 청크는 [10명 페르소나]. 각 페르소나는 persona 정보 + versions 별 vote/reason."""
    sub = df[(df["model"] == model) & df["voter_context_version"].isin(versions)]
    sub = sub.drop_duplicates(["persona_uuid", "voter_context_version"], keep="last")
    # 모든 버전이 다 있는 페르소나만
    counts = sub.groupby("persona_uuid")["voter_context_version"].nunique()
    full = counts[counts == len(versions)].index
    sub = sub[sub["persona_uuid"].isin(full)].sort_values("persona_uuid")
    print(f"  {model}: {len(full)}명 × {len(versions)}버전")

    persona_cols = [
        "persona_uuid", "sex", "age", "marital_status", "family_type",
        "housing_type", "education_level", "bachelors_field", "occupation",
        "province", "district", "persona_summary",
    ]
    pi_map = dict(zip(interests["persona_uuid"], interests["political_interest"]))

    personas = []
    for uid, group in sub.groupby("persona_uuid"):
        row0 = group.iloc[0]
        p = {c: (str(row0[c]) if pd.notna(row0[c]) else "") for c in persona_cols}
        p["political_interest"] = pi_map.get(uid, "")
        p["votes"] = {}
        for _, r in group.iterrows():
            v = r["voter_context_version"]
            p["votes"][v] = {
                "vote": str(r["vote"]) if pd.notna(r["vote"]) else None,
                "reason": str(r["reason"]) if pd.notna(r["reason"]) else "",
            }
        personas.append(p)

    # 10개 청크로 분할
    per_chunk = max(1, (len(personas) + chunks_n - 1) // chunks_n)
    return [personas[i : i + per_chunk] for i in range(0, len(personas), per_chunk)]


GPT_VERSIONS = ["v2", "v3", "v4", "v5", "v6"]
HAIKU_VERSIONS = ["v4", "v5", "v6"]

print("GPT 청크 생성:")
gpt_chunks = build_chunks(GPT_MODEL, GPT_VERSIONS)
print("HAIKU 청크 생성:")
haiku_chunks = build_chunks(HAIKU_MODEL, HAIKU_VERSIONS)

print(f"\nGPT: {len(gpt_chunks)}개 청크 (각 {[len(c) for c in gpt_chunks]})")
print(f"HAIKU: {len(haiku_chunks)}개 청크 (각 {[len(c) for c in haiku_chunks]})")

In [ ]:
# 각 청크를 JSON 파일로 저장 (서브에이전트가 읽을 수 있게)
def save_chunks(chunks: list[list[dict]], prefix: str) -> list[Path]:
    paths = []
    for i, chunk in enumerate(chunks, 1):
        p = EVAL_DIR / f"{prefix}_{i:03d}_input.json"
        p.write_text(json.dumps(chunk, ensure_ascii=False, indent=2), encoding="utf-8")
        paths.append(p)
    return paths

gpt_paths = save_chunks(gpt_chunks, "gpt")
haiku_paths = save_chunks(haiku_chunks, "haiku")
for p in gpt_paths + haiku_paths:
    print(p, p.stat().st_size, "bytes")

## 서브에이전트 dispatch

이 노트북에서는 직접 dispatch 안 하고, **Claude Code 채팅에서 Agent 도구로 dispatch** 한다.

각 에이전트에게 `eval/{model}_{NNN}_input.json` 을 읽고 채점하도록 지시. 결과는 `eval/{model}_{NNN}_eval.json` + `eval/{model}_{NNN}_eval.md` 로 저장.

**프롬프트 본문은 다음 셀의 `EVAL_PROMPT` 참조.**

In [ ]:
EVAL_PROMPT = """
당신은 페르소나-LLM 투표 시뮬레이션 결과를 평가하는 분석가다.

**입력 파일을 반드시 읽고 평가하라**:
{INPUT_PATH}

이 파일에는 N명의 페르소나가 있고, 각 페르소나마다 v2/v3/v4/v5/v6 컨텍스트(또는 v4/v5/v6)에서의 vote+reason 이 들어있다.

## 컨텍스트 변화 요약 (각 단계의 의도)
- **v2 → v3**: voter_context 에서 "지역별 통상 정치 지형" + "세대별 일반 경향" 두 섹션을 제거.
  → reason 안에서 "호남이라서", "60대니까" 같은 인구통계적 일반화가 줄어들었는가?
- **v3 → v4**: 페르소나 카드에 1인칭 정치 이해관계 블록(평균 633자)을 추가. 각자가 자기 입장에서 2026년 정치 이슈를 검색·정리한 텍스트.
  → reason 이 그 정치 이해관계 블록의 구체 내용을 직접 인용·참조하는가?
- **v4 → v5**: voter_context 에 정당별 정책 카드 6개 추가, system_prompt 에 "이해관계 ↔ 정당 강령 매칭이 1순위" 명시, 무당층 억제.
  → reason 이 정당의 구체 정책명(검찰개혁/연금/부동산/노동 등)을 짝지어 언급하는가?
- **v5 → v6**: 권력 라벨 뒤집기 — 가상 대통령 "장재현"(국민의힘), 의석 국힘 160 / 민주 107.
  → reason 이 가상 권력 시그널(장재현/여당/의석)을 명시적으로 인지·반영했는가?

## 평가 항목
각 페르소나에 대해 다음을 평가하라.

1. **votes**: [v2, v3, v4, v5, v6] 정당명 리스트 (해당 버전이 없으면 null)
2. **n_changes**: 인접 버전 사이에서 정당이 바뀐 횟수 (0~4)
3. **trajectory**: 
   - `stable` — 변화 0회
   - `drift` — 1~2회 점진 이동
   - `swing` — 3~4회 잦은 변화
4. **biggest_jump**: 가장 큰/대표적 이동 시점 (예: "v3→v4"). 변화 없으면 null.
5. **fidelity** (각 항목 1~5점, 1=무반영, 3=부분, 5=명시적·정확):
   - `v2_v3_demographic_drop`: v2 reason 의 인구통계 일반화가 v3 에서 줄었나 (v2/v3 둘 다 있을 때만)
   - `v3_v4_pi_citation`: v4 reason 이 정치 이해관계 블록을 직접 인용했나 (v3/v4 둘 다 있을 때만)
   - `v4_v5_policy_match`: v5 reason 이 구체 정당 정책을 매칭 언급했나 (v4/v5 둘 다)
   - `v5_v6_power_signal`: v6 reason 이 권력 시그널(장재현/국힘여당/160석)을 인지·반영했나 (v5/v6 둘 다)
   - 해당 단계가 없으면 항목을 null 로.
6. **reason_quality_trend**: `improving` (점점 정교) / `stable` / `shallow` (피상적)
7. **narrative**: 한국어 1~2문장 요약. 직업·지역 등 페르소나 디테일과 함께 어떤 패턴을 보였는지.

## 출력
다음 두 파일을 작성하라:

**(1) `{OUTPUT_JSON}`** — 다음 형식의 JSON 배열 (페르소나당 1개 객체):
```json
[
  {{
    "persona_uuid": "...",
    "votes": ["민주당", "민주당", "민주당", "민주당", "국민의힘"],
    "n_changes": 1,
    "trajectory": "drift",
    "biggest_jump": "v5→v6",
    "fidelity": {{
      "v2_v3_demographic_drop": 4,
      "v3_v4_pi_citation": 5,
      "v4_v5_policy_match": 4,
      "v5_v6_power_signal": 5
    }},
    "reason_quality_trend": "improving",
    "narrative": "포항 제도사. v2~v5 정책·연금 의제로 민주를 유지하다가 v6 권력 라벨 변화에 직접 반응해 국힘으로 점프."
  }},
  ...
]
```

**(2) `{OUTPUT_MD}`** — 한국어 분석 요약 마크다운:
- 청크 N명의 trajectory 분포 (stable/drift/swing 카운트)
- 각 fidelity 항목의 평균 점수와 분포
- 가장 흥미로운 사례 1~3건 narrative
- 패턴 관찰 (예: "권력 시그널에 즉시 반응한 페르소나는 X명, 모두 OO 직업군")

정확하고 간결하게. JSON은 반드시 valid 해야 한다 (코드블록 없이 .json 파일 그 자체).
"""

# 프린트해서 카피 가능하게
print("=" * 80)
print("GPT 청크 dispatch 명령 예시:")
print("=" * 80)
for p in gpt_paths[:1]:
    out_json = p.with_name(p.stem.replace("_input", "_eval") + ".json")
    out_md = p.with_name(p.stem.replace("_input", "_eval") + ".md")
    print(f"\n# {p.name}")
    print(EVAL_PROMPT.format(INPUT_PATH=str(p).replace('\\\\','/'), OUTPUT_JSON=str(out_json).replace('\\\\','/'), OUTPUT_MD=str(out_md).replace('\\\\','/')))

## 결과 집계 (모든 에이전트 끝난 후 실행)

In [ ]:
import json
from collections import Counter
import statistics

def aggregate(prefix: str) -> dict:
    files = sorted(EVAL_DIR.glob(f"{prefix}_*_eval.json"))
    all_evals = []
    for f in files:
        try:
            data = json.loads(f.read_text(encoding="utf-8"))
            if isinstance(data, list):
                all_evals.extend(data)
        except Exception as e:
            print(f"PARSE ERR {f.name}: {e}")

    print(f"{prefix}: {len(files)} 파일, 총 {len(all_evals)} 페르소나 평가")
    if not all_evals:
        return {}

    traj = Counter(e.get("trajectory") for e in all_evals)
    n_changes = Counter(e.get("n_changes") for e in all_evals)
    trends = Counter(e.get("reason_quality_trend") for e in all_evals)
    biggest = Counter(e.get("biggest_jump") for e in all_evals if e.get("biggest_jump"))

    fid_avg = {}
    for key in ["v2_v3_demographic_drop", "v3_v4_pi_citation", "v4_v5_policy_match", "v5_v6_power_signal"]:
        vals = [e["fidelity"][key] for e in all_evals if e.get("fidelity") and isinstance(e["fidelity"].get(key), (int, float))]
        if vals:
            fid_avg[key] = {"mean": round(statistics.mean(vals), 2), "n": len(vals), "hist": dict(Counter(vals))}

    # vote 경로 (4-튜플 시퀀스)
    paths = Counter(" → ".join(str(v) for v in e["votes"]) for e in all_evals if e.get("votes"))

    out = {
        "n_personas": len(all_evals),
        "trajectory": dict(traj),
        "n_changes": dict(n_changes),
        "reason_quality_trend": dict(trends),
        "biggest_jump": dict(biggest),
        "fidelity": fid_avg,
        "top_paths": dict(paths.most_common(10)),
    }
    print(json.dumps(out, ensure_ascii=False, indent=2))
    return out

gpt_summary = aggregate("gpt")
haiku_summary = aggregate("haiku")

In [ ]:
# 최종 분석 보고서를 doc/ 에 저장
DOC = BASE.parent / "doc"
if gpt_summary:
    (DOC / "v2_v6_consistency_analysis_gpt.json").write_text(
        json.dumps(gpt_summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
if haiku_summary:
    (DOC / "v2_v6_consistency_analysis_haiku.json").write_text(
        json.dumps(haiku_summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
print("saved.")